In [2]:
# Cell 1 — Install PEFT and datasets
#%pip install -q peft==0.4.0 datasets

# transformers is usually already installed in Colab; if not, uncomment:
#%pip install -q transformers

# Create cache folder for outputs
!mkdir -p cache


In [ ]:
# Cell 2 — Imports and base model

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os
import time

# Choose the base causal LM (as requested)
model_name = "bigscience/bloomz-560m"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# Ensure we have a pad token (BLOOM-style models sometimes need this)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Force model to use CPU (as in DI notebooks)
device = torch.device("cpu")
foundation_model.to(device)

print("Base model loaded on:", device)


In [ ]:
# Cell 3 — Load and preprocess the dataset

from IPython.display import display

# Load the training split of the Abirate/english_quotes dataset
raw_data = load_dataset("Abirate/english_quotes", split="train")

print("Total examples in train split:", len(raw_data))

# Take a 10% random sample
sample_size = max(1, int(0.1 * len(raw_data)))
data = raw_data.shuffle(seed=42).select(range(sample_size))

print("Sampled examples:", len(data))

# Tokenize the "quote" field
def tokenize_function(samples):
    return tokenizer(
        samples["quote"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

data = data.map(tokenize_function, batched=True)

# For Trainer, we keep only tensor-features that the model needs
columns_to_keep = ["input_ids", "attention_mask"]
if "labels" in data.column_names:
    # Not expected here, but just in case
    columns_to_keep.append("labels")

# For causal LM, labels = input_ids (standard language modeling)
data = data.map(
    lambda samples: {"labels": samples["input_ids"]},
    batched=True,
)

# Select subset for quick preview
train_sample = data.select(range(min(5, len(data))))
display(train_sample)

print("Final columns:", data.column_names)


In [ ]:
# Cell 4 — Configure LoRA (PEFT)

import peft
from peft import LoraConfig, get_peft_model

# For BLOOM-style models, typical target modules include:
# "query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"
target_modules = ["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"]

lora_config = LoraConfig(
    r=8,                      # rank of LoRA matrices
    lora_alpha=8,             # scaling factor
    target_modules=target_modules,
    lora_dropout=0.1,         # dropout on LoRA updates
    bias="none",              # do not train bias terms
    task_type="CAUSAL_LM"     # because we are doing language modeling
)

# Add LoRA adapters on top of the foundation model
peft_model = get_peft_model(foundation_model, lora_config)
peft_model.to(device)

print(peft_model.print_trainable_parameters())


In [ ]:
# Cell 5 — TrainingArguments and Trainer

import transformers
from transformers import TrainingArguments, Trainer

output_directory = os.path.join("cache", "peft_lab_outputs")

training_args = TrainingArguments(
    report_to="none",           # disable wandb etc.
    output_dir=output_directory,
    auto_find_batch_size=True,  # let Trainer pick a suitable batch size
    learning_rate=3e-2,         # higher LR for LoRA fine-tuning (as suggested)
    num_train_epochs=1,         # keep small for the exercise
    use_cpu=True,               # force CPU training
    logging_steps=10,
    save_strategy="no"          # we will save manually after training
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,  # use our tokenized dataset
    data_collator=transformers.DataCollatorForLanguageModeling(
        tokenizer,
        mlm=False  # causal language modeling, not masked LM
    )
)

trainer


In [ ]:
# Cell 6 — Train

trainer.train()


In [ ]:
# Cell 7 — Save PEFT model

time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
os.makedirs(peft_model_path, exist_ok=True)

# Save only the LoRA adapter weights (PEFT model)
trainer.model.save_pretrained(peft_model_path)
# Save tokenizer as well so we can reload everything later
tokenizer.save_pretrained(peft_model_path)

print("PEFT model saved to:", peft_model_path)


In [ ]:
# Cell 8 — Reload base model + LoRA weights for inference

from peft import PeftModel

# Reload the base foundation model
base_model_for_inference = AutoModelForCausalLM.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model_for_inference.to(device)

# Attach the saved LoRA adapter
peft_model_infer = PeftModel.from_pretrained(
    base_model_for_inference,
    peft_model_path,
    is_trainable=False  # we are not training anymore
)

peft_model_infer.to(device)
peft_model_infer.eval()

print("Loaded PEFT model for inference from:", peft_model_path)


In [ ]:
# Cell 9 — Inference / generation

prompt_text = "Two things are infinite: "
inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = peft_model_infer.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print("=== Generated text ===")
print(generated_text)
